# Distilling Wikontic Full Pipeline into SmolLM2-1.7B

**Goal:** Train a small local model (SmolLM2-1.7B-Instruct) to replace the entire Wikontic pipeline -- taking raw text and producing ontology-aligned triplets in a single forward pass.

**Data:**
- datasets/hotpotqa200.json -- source texts (200 samples, ~10 paragraphs each)
- datasets/kg_dump_hotpot_gpt4_1_onto_triplets.json -- ground-truth ontology-aligned triplets produced by GPT-4.1 via the full Wikontic pipeline

**Approach:** QLoRA fine-tuning with trl.SFTTrainer. Each training example is a chat-formatted (text -> triplets) pair derived from the same prompt that Wikontic uses for extraction.

In [1]:
# Install missing dependencies (trl, peft, bitsandbytes, datasets)
! pip install -q trl peft bitsandbytes datasets



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load and Prepare Data

In [1]:
import json
from pathlib import Path
import os
import sys
os.environ['PYTHONUTF8'] = '1'

HOTPOT_PATH = Path("../datasets/hotpotqa200.json")
DUMP_PATH   = Path("../datasets/kg_dump_hotpot_gpt4_1_onto_triplets.json")
SYSTEM_PROMPT_PATH = Path("../src/wikontic/utils/prompts/triplet_extraction/propmt_1_types_qualifiers.txt")

OUTPUT_DIR = Path("./data")
TRAIN_OUT = OUTPUT_DIR / "train.jsonl"
VAL_OUT   = OUTPUT_DIR / "val.jsonl"

CHECKPOINT_DIR = "./checkpoints/wikontic-full-pipeline"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load hotpotqa -- indexed by sample_id
with open(HOTPOT_PATH) as f:
    hotpot = {s["_id"]: s for s in json.load(f)}

# Load kg_dump -- same structure: {sample_id: {source_id: {triplets, ...}}}
with open(DUMP_PATH) as f:
    dump = json.load(f)

print(f"HotPotQA samples: {len(hotpot)}")
print(f"KG dump samples:   {len(dump)}")
print(f"Overlapping IDs:   {len(set(hotpot) & set(dump))}")

# Load system prompt
SYSTEM_PROMPT = open(SYSTEM_PROMPT_PATH, encoding="utf-8").read()
print(f"\nSystem prompt length: {len(SYSTEM_PROMPT)} chars")



HotPotQA samples: 200
KG dump samples:   200
Overlapping IDs:   200

System prompt length: 4034 chars


## 2. Inspect Data Structure

In [2]:
# Inspect a single example
sample_id = list(dump.keys())[0]
sample = hotpot[sample_id]
entry  = dump[sample_id]["0"]   # first paragraph

print("=== HotPotQA sample ===")
print("Question:", sample["question"])
print("Answer:",   sample["answer"])
print("Context count:", len(sample["context"]))
title, chunks = sample["context"][0]
print("First paragraph title:", title)
print("First paragraph text:", " ".join(chunks)[:200], "...")

print("\n=== KG dump for this paragraph ===")
print("Triplets count:", len(entry["triplets"]))
for t in entry["triplets"][:3]:
    print(" -", t)

=== HotPotQA sample ===
Question: Which of these universities, Northwestern University or Johns Hopkins University, have a campus outside of the United States territories?
Answer: with other campuses located in Chicago and Doha, Qatar
Context count: 10
First paragraph title: Northwestern University
First paragraph text: Northwestern University (NU) is a private research university based in Evanston, Illinois, with other campuses located in Chicago and Doha, Qatar, and academic programs and facilities in Washington, D ...

=== KG dump for this paragraph ===
Triplets count: 6
 - {'object': 'private research university', 'object_type': 'educational institution', 'relation': 'instance of', 'subject': 'Northwestern University', 'subject_type': 'university', 'qualifiers': []}
 - {'object': 'Evanston, Illinois', 'object_type': 'city', 'relation': 'headquarters location', 'subject': 'Northwestern University', 'subject_type': 'university', 'qualifiers': []}
 - {'object': 'Chicago', 'object_type

## 3. Build Training Examples

In [3]:
def build_example(text: str, triplets: list, system_prompt: str) -> dict:
    """Format one text-triplets pair as chat messages for SFT training."""
    assistant_content = json.dumps({"triplets": triplets}, ensure_ascii=False)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Text: "{text}"'},
        {"role": "assistant", "content": assistant_content},
    ]
    return {"messages": messages}


examples = []
skipped_empty = 0

for sample_id, source_dict in dump.items():
    if sample_id not in hotpot:
        continue
    sample = hotpot[sample_id]
    context = sample["context"]   # list of [title, [text_segments]]

    for sid_str, entry in source_dict.items():
        sid = int(sid_str)
        if sid >= len(context):
            continue

        title, text_segments = context[sid]
        text = " ".join(text_segments).strip()
        triplets = entry.get("triplets", [])

        if not text or not triplets:
            skipped_empty += 1
            continue

        examples.append(build_example(text, triplets, SYSTEM_PROMPT))

print(f"Total examples: {len(examples)}")
print(f"Skipped (empty text or triplets): {skipped_empty}")


Total examples: 1999
Skipped (empty text or triplets): 0


In [4]:
# Basic statistics
def count_triplets(example_text):
    """Extract JSON object from text and return triplet count."""
    try:
        start = example_text.rfind('{"triplets":')
        if start < 0:
            return 0
        json_start = example_text.find('[', start)
        if json_start < 0:
            return 0
        depth = 1
        pos = json_start + 1
        while pos < len(example_text) and depth > 0:
            c = example_text[pos]
            if c == '[':
                depth += 1
            elif c == ']':
                depth -= 1
            pos += 1
        if pos < len(example_text) and example_text[pos] == '}':
            candidate = example_text[start:pos + 1]
            return len(json.loads(candidate)['triplets'])
        return 0
    except Exception:
        return 0

triplet_counts = [count_triplets(e['messages'][2]['content']) for e in examples]
non_zero = sum(1 for c in triplet_counts if c > 0)
print(f"Min / Max / Avg triplets per example: {min(triplet_counts)} / {max(triplet_counts)} / {sum(triplet_counts)/len(triplet_counts):.1f}")
print(f"Non-zero: {non_zero} / {len(triplet_counts)}")

# Show one formatted example
ex = examples[10]
print()
print("=== Example #10 (assistant output) ===")
print(ex['messages'][2]['content'][:200])


Min / Max / Avg triplets per example: 1 / 48 / 11.2
Non-zero: 1999 / 1999

=== Example #10 (assistant output) ===
{"triplets": [{"subject_type": "human", "object_type": "social group", "relation": "part of", "subject": "K. Ravindran Nair", "object": "rich family", "qualifiers": []}, {"subject_type": "human", "obj


## 4. Train / Val Split and Save

In [5]:
import random

random.seed(42)
random.shuffle(examples)

val_size = max(1, int(len(examples) * 0.1))
val_examples   = examples[:val_size]
train_examples = examples[val_size:]

def write_jsonl(path: Path, items: list):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

write_jsonl(TRAIN_OUT, train_examples)
write_jsonl(VAL_OUT,   val_examples)

print(f"Train: {len(train_examples)} | Val: {len(val_examples)}")
print(f"Saved to {TRAIN_OUT} and {VAL_OUT}")

Train: 1800 | Val: 199
Saved to data\train.jsonl and data\val.jsonl


## 5. Load Model -- SmolLM2-1.7B-Instruct with QLoRA

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

BASE_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# 4-bit NF4 quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

# LoRA adapter
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

D:\PycharmProjects\Wikontic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading base model...


Loading weights: 100%|██████████| 218/218 [00:03<00:00, 63.83it/s]


trainable params: 3,145,728 || all params: 1,714,522,112 || trainable%: 0.1835


## 6. Configure and Run SFT Training

In [8]:
# Fix encoding issue on Windows before importing trl
from pathlib import Path
import sys
sys.path.insert(0, str(Path(".").resolve()))
import trl_utf8_fix  # noqa: E402

from trl import SFTTrainer
from trl import SFTConfig
from datasets import load_dataset

CHECKPOINT_DIR = "./checkpoints/wikontic-full-pipeline"

# Load datasets
from datasets import Dataset, DatasetDict

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

ds = DatasetDict({
    "train": Dataset.from_list(load_jsonl(TRAIN_OUT)),
    "val":   Dataset.from_list(load_jsonl(VAL_OUT)),
})
print(f"Train: {len(ds['train'])} | Val: {len(ds['val'])}")


# Pre-process: split each example into prompt / completion columns.
# This is required for completion_only_loss=True to work (it needs a
# prompt-completion dataset, not a raw text formatter).
def split_prompt_completion(example):
    messages = example["messages"]
    # Prompt: all turns up to (but not including) the assistant response,
    # with add_generation_prompt=True so the assistant header is included.
    prompt = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )
    # Full formatted conversation without a trailing generation prompt.
    full = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    # Completion is everything after the prompt prefix.
    completion = full[len(prompt):]
    # Ensure the completion ends with EOS so the model learns to stop.
    if not completion.endswith(tokenizer.eos_token):
        completion += tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}


ds["train"] = ds["train"].map(split_prompt_completion, remove_columns=["messages"])
ds["val"]   = ds["val"].map(split_prompt_completion, remove_columns=["messages"])
print("Prepared prompt/completion dataset.")

# Drop any examples where completion is empty (would produce NaN in eval loss)
def has_nonempty_completion(example):
    return bool(example.get("completion", "").strip())

ds["train"] = ds["train"].filter(has_nonempty_completion)
ds["val"]   = ds["val"].filter(has_nonempty_completion)
print(f"After filtering empty completions -- Train: {len(ds['train'])} | Val: {len(ds['val'])}")


# Training arguments with completion-only loss
# This ensures model only learns to predict assistant responses, not system/user prompts
training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,       # effective batch = 32
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    max_grad_norm=0.3,
    bf16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    completion_only_loss=True,  # KEY: Only train on assistant responses
    seed=42,
)

# Clamp extreme logits during eval to avoid bf16 overflow → NaN loss
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Cast to fp32 and clamp to a safe range before metrics/loss computation
    return logits.float().clamp(min=-50, max=50)

# SFT Trainer - model is already a PeftModel
# NOTE: We do NOT pass formatting_func because the dataset is already in
# prompt-completion format, which SFTTrainer natively supports for
# completion-only loss.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    processing_class=tokenizer,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
)

print("Starting training...")
trainer.train()




Train: 1800 | Val: 199


Map: 100%|██████████| 199/199 [00:00<00:00, 18560.11 examples/s]


Prepared prompt/completion dataset.


Tokenizing eval dataset: 100%|██████████| 199/199 [00:00<00:00, 396.11 examples/s]


Starting training...


Epoch,Training Loss,Validation Loss
1,1.082464,nan
2,0.577714,nan
3,0.449056,nan


TrainOutput(global_step=171, training_loss=0.7492968443541499, metrics={'train_runtime': 21707.2582, 'train_samples_per_second': 0.249, 'train_steps_per_second': 0.008, 'total_flos': 5.35439622537216e+16, 'train_loss': 0.7492968443541499})

## 7. Save Model

In [9]:
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
print(f"Model saved to {CHECKPOINT_DIR}")

Model saved to ./checkpoints/wikontic-full-pipeline


## 8. Quick Validation -- Inference on a Few Examples

In [10]:
CHECKPOINT_DIR+"/checkpoint-57"

'./checkpoints/wikontic-full-pipeline/checkpoint-57'

In [7]:
from peft import PeftModel
import re


def extract_triplets(text):
    """Extract triplets from model output, handling complete and partial JSON."""
    # Attempt 1: Clean trailing garbage and parse full JSON
    cleaned = text.rstrip(".")
    try:
        data = json.loads(cleaned)
        if isinstance(data, dict) and "triplets" in data:
            return data
    except (json.JSONDecodeError, TypeError):
        pass

    # Attempt 2: Find JSON object with a "triplets" key anywhere in the text
    # Greedy match from first '{' to last '}' that makes a valid dict
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        for candidate in [text[start:end+1], text[start:end+1].rstrip(".")]:
            try:
                data = json.loads(candidate)
                if isinstance(data, dict) and "triplets" in data:
                    return data
            except (json.JSONDecodeError, TypeError):
                pass

    # Attempt 3: Use brace-depth extraction for nested structures
    candidates = []
    for i, ch in enumerate(text):
        if ch == "{":
            depth = 1
            for j in range(i + 1, len(text)):
                if text[j] == "{":
                    depth += 1
                elif text[j] == "}":
                    depth -= 1
                    if depth == 0:
                        candidates.append(text[i:j+1])
                        break

    triplets = []
    seen = set()
    for candidate in candidates:
        # Quick heuristic: must mention subject, relation, object
        if '"subject"' not in candidate or '"relation"' not in candidate or '"object"' not in candidate:
            continue
        try:
            obj = json.loads(candidate)
            if isinstance(obj, dict) and all(k in obj for k in ("subject", "relation", "object")):
                key = (obj.get("subject"), obj.get("relation"), obj.get("object"))
                if key not in seen:
                    seen.add(key)
                    triplets.append(obj)
        except (json.JSONDecodeError, TypeError):
            pass

    if triplets:
        return {"triplets": triplets}

    return None

def inference(text, max_new_tokens=2048):
    """Run the student model on a single text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Text: "{text}"'},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(output[0], skip_special_tokens=True)

    print("Full output:", full_output)
    
    # Extract assistant portion (after assistant role marker)
    marker = "assistant\n"
    marker_pos = full_output.find(marker)
    if marker_pos >= 0:
        answer = full_output[marker_pos + len(marker):].strip()
    else:
        # Fallback: remove prompt portion
        answer = full_output[len(prompt):].strip()

    # Remove any trailing system/user turns if model hallucinated them
    for turn_marker in ["<|im_start|>user", "<|im_start|>system"]:
        pos = answer.find(turn_marker)
        if pos >= 0:
            answer = answer[:pos].strip()
    
    print("Model answer (first 500 chars):\n", answer[:500])
    
    return extract_triplets(answer)


pred_results = []
# Run on 3 val examples
for i in range(3):
    ex = val_examples[i]
    # Extract text from messages
    text = ex["messages"][1]["content"].replace('Text: "', '').rstrip('"')
    
    # Extract ground truth triplets
    gt_text = ex["messages"][2]["content"]
    # Remove EOS token if present
    gt_text = gt_text.replace(tokenizer.eos_token, "")
    gt_parsed = extract_triplets(gt_text)
    gt_triplets = gt_parsed['triplets'] if gt_parsed else []

    pred = inference(text)

    pred_results.append(pred)
    print(f"--- Example {i} ---")
    print(f"Ground truth triplets ({len(gt_triplets)}):")
    for t in gt_triplets[:3]:
        print(f"  {t['subject']} | {t['relation']} | {t['object']}")
    print(f"\nPredicted (valid JSON: {pred is not None}):")
    if pred and "triplets" in pred:
        for t in pred["triplets"][:3]:
            print(f"  {t.get('subject','?')} | {t.get('relation','?')} | {t.get('object','?')}")
    else:
        print("  (failed to parse or no triplets)")
    print()


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Full output: system
You are an algorithm designed to extract structured knowledge from texts to build a Wikidata-like knowledge graph. A knowledge graph consists of **triplets** in the format (subject, relation, object), where:

- **Subject**: A named entity or a concept that describes a group of people, events, or any abstract objects that serves as the source of the relation.
- **Relation**: A Wikidata-style predicate that connects the subject and object.
- **Object**: A named entity or a concept that describes a group of people, events, or any abstract objects that is related to the subject.

Additionally, some triplets may have **qualifiers** that provide more context (e.g., date, place, or other attributes). Qualifiers should have relations and object like triplets do, but instead of subject their relation connects an object and the triplet qualifier belongs to. **Qualifiers must always be attached to a triplet** and never exist as standalone triplets.

You will receive a text lab